# 07 MLP Controlled Trial Comparison

Sequential screening is interpretable but order-dependent and can miss interactions. This notebook builds a conservative HPO domain instead of claiming a global optimum.


## 1. Imports and Paths


In [1]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the project files, cached MFCC features, manifests, models, figures,
# and metric outputs live in Google Drive during Colab runs. The /content/drive path
# only represents the real MyDrive files after drive.mount("/content/drive") succeeds.
# Important: the project root should be the folder that contains Data, Model Variants,
# and outputs. For this project, that expected Colab folder is the INM701 folder below.
COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount(str(COLAB_DRIVE_MOUNT_POINT))
    print("Google Colab detected. Google Drive mounted.")

    current_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT")
    current_data_dir_exists = bool(current_project_root) and (Path(current_project_root) / "Data").exists()
    expected_data_dir_exists = (EXPECTED_COLAB_PROJECT_ROOT / "Data").exists()

    # Purpose: Keep a valid user-provided project root, but repair stale runtime state
    # if a previous cell pointed INTRO_AI_PROJECT_ROOT somewhere that does not contain Data.
    if current_data_dir_exists:
        print("INTRO_AI_PROJECT_ROOT already points to a folder with Data, so it was kept.")
    elif expected_data_dir_exists:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.")
    elif not current_project_root:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT was not set, so it now points to the expected INM701 folder.")
    else:
        print("INTRO_AI_PROJECT_ROOT was kept, but Data was not found there or in the expected INM701 folder.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

active_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT", "not set")
print("INTRO_AI_PROJECT_ROOT:", active_project_root)
if active_project_root != "not set":
    active_project_root = Path(active_project_root)
    print("Project root exists:", active_project_root.exists())
    print("Expected Data folder:", active_project_root / "Data")
    print("Data folder exists:", (active_project_root / "Data").exists())


Mounted at /content/drive
Google Colab detected. Google Drive mounted.
INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.
INTRO_AI_PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Project root exists: True
Expected Data folder: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Data
Data folder exists: True


In [2]:
# Purpose: 1. Imports and Paths.
import json
import os
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
CONFIGS_DIR = OUTPUT_DIR / "configs"
TABLES_DIR = OUTPUT_DIR / "tables"
METRICS_DIR = OUTPUT_DIR / "metrics"
# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [CONFIGS_DIR, TABLES_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def load_json(path):
    # Purpose: Keeps the load_json helper isolated so later notebook cells can call it consistently.
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(payload, path):
    # Purpose: Keeps the save_json helper isolated so later notebook cells can call it consistently.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def normalise_handoff_config(config):
    # Purpose: Normalises a config loaded from a JSON string or CSV row before it is reused
    # by the next notebook in the MLP chain. This keeps rebuilt handoff files compatible
    # with configs produced by a full training run.
    normalised = dict(config)
    hidden_units = normalised.get("hidden_units", [])
    if isinstance(hidden_units, str):
        hidden_units = json.loads(hidden_units)
    normalised["hidden_units"] = [int(value) for value in hidden_units]
    normalised["activation"] = str(normalised["activation"])
    normalised["optimizer"] = str(normalised["optimizer"])
    normalised["dropout"] = float(normalised["dropout"])
    normalised["learning_rate"] = float(normalised["learning_rate"])
    normalised["batch_size"] = int(normalised["batch_size"])
    normalised["loss"] = normalised.get("loss", "binary_crossentropy")
    normalised["threshold"] = float(normalised.get("threshold", 0.5))
    return normalised


def first_existing_results_path(candidate_paths):
    # Purpose: Supports the normal result filename and any documented alias without changing
    # where the notebook saves new results. The first existing CSV is used for rebuilding.
    paths = [Path(path) for path in candidate_paths]
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def rebuild_selected_config_from_results(results_path, selected_path, selected_by_stage):
    # Purpose: Rebuilds a missing selected_config JSON from an existing validation-results
    # CSV. This avoids rerunning training just to recreate the JSON handoff file.
    results_path = Path(results_path)
    selected_path = Path(selected_path)
    if not results_path.exists():
        print("Cannot rebuild selected config because result CSV is missing:", results_path)
        return None

    results_df = pd.read_csv(results_path)
    required_columns = {"config_json", "validation_macro_f1", "best_validation_loss"}
    missing_columns = required_columns - set(results_df.columns)
    if missing_columns:
        raise RuntimeError(f"Cannot rebuild {selected_path.name}; missing columns in {results_path.name}: {sorted(missing_columns)}")
    if results_df.empty:
        raise RuntimeError(f"Cannot rebuild {selected_path.name}; result CSV is empty: {results_path}")

    best_row = results_df.sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).iloc[0]
    selected = normalise_handoff_config(json.loads(best_row["config_json"]))
    selected.update({
        "selected_by_stage": selected_by_stage,
        "selection_metric": "validation_macro_f1",
        "validation_macro_f1": float(best_row["validation_macro_f1"]),
        "best_validation_loss": float(best_row["best_validation_loss"]),
        "rebuilt_from_results_csv": str(results_path),
    })
    save_json(selected, selected_path)
    print("Rebuilt selected config from existing results:", selected_path)
    return selected


def ensure_selected_config_from_results(selected_path, results_path, selected_by_stage):
    # Purpose: Repairs a missing selected config before comparison or search-space creation.
    selected_path = Path(selected_path)
    results_path = Path(results_path)
    if selected_path.exists():
        return load_json(selected_path)
    if results_path.exists():
        return rebuild_selected_config_from_results(results_path, selected_path, selected_by_stage)
    print("Selected config missing:", selected_path)
    print("Result CSV also missing, so config cannot be rebuilt:", results_path)
    return None



## 2. Load Results and Build Search Space


In [3]:
# Purpose: Loads controlled-screening result tables, rebuilds any missing JSON handoff
# files from those tables, and creates the compact search space used by the HPO notebooks.
files = {
    "activation": TABLES_DIR / "02_activation_results.csv",
    "optimizer": TABLES_DIR / "03_optimizer_results.csv",
    "dropout": TABLES_DIR / "04_dropout_results.csv",
    "architecture": first_existing_results_path([TABLES_DIR / "05_architecture_results.csv", TABLES_DIR / "05_hidden_units_results.csv"]),
    "learning_rate": TABLES_DIR / "06_learning_rate_results.csv",
    "batch_size": TABLES_DIR / "06_batch_size_results.csv",
}
results = {name: pd.read_csv(path) for name, path in files.items() if path.exists()}
# Purpose: Lists any missing controlled-screening result files so the tester knows which
# previous notebook still needs to be run.
for name, path in files.items():
    if name not in results:
        print("Missing", name, path, "[RESULT TO BE INSERTED AFTER FINAL RUN]")

def parse_hidden_units(value):
    # Purpose: Converts hidden-unit values from CSV text back into the list format expected
    # by the MLP builder.
    if isinstance(value, list):
        return value
    return [int(part.strip()) for part in str(value).strip("[]").split(",") if part.strip()]

def top_values(df, column, keep):
    # Purpose: Keeps the best distinct values from a controlled-screening table using the
    # same validation macro-F1 then validation-loss selection rule.
    ranked = df.sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True])
    values = []
    for value in ranked[column].tolist():
        if column == "hidden_units":
            value = parse_hidden_units(value)
        if value not in values:
            values.append(value)
        if len(values) >= keep:
            break
    return values

def domain_size(space):
    # Purpose: Counts the number of possible HPO configurations implied by the retained
    # search-space values.
    total = 1
    for values in space.values():
        total *= len(values)
    return total


# Build a compact controlled-screening summary for the report. This is validation evidence only.
summary_rows = []
baseline_metrics_path = METRICS_DIR / "baseline_validation_metrics.json"
if baseline_metrics_path.exists():
    baseline_metrics = load_json(baseline_metrics_path)
    summary_rows.append({
        "stage": "baseline",
        "candidate": "fixed_reference",
        "varied_factor": "none",
        "validation_macro_f1": baseline_metrics.get("validation_macro_f1"),
        "best_validation_loss": baseline_metrics.get("best_validation_loss"),
        "runtime_seconds": baseline_metrics.get("runtime_seconds"),
        "interpretation": "starting point, not an optimum",
    })

factor_columns = {
    "activation": "activation",
    "optimizer": "optimizer",
    "dropout": "dropout",
    "architecture": "hidden_units",
    "learning_rate": "learning_rate",
    "batch_size": "batch_size",
}
# Purpose: Walks through each controlled-screening table and records one readable summary
# row per tested candidate.
for stage, df in results.items():
    candidate_column = factor_columns[stage]
    for _, row in df.iterrows():
        summary_rows.append({
            "stage": stage,
            "candidate": row.get(candidate_column),
            "varied_factor": candidate_column,
            "validation_macro_f1": row.get("validation_macro_f1"),
            "best_validation_loss": row.get("best_validation_loss"),
            "runtime_seconds": row.get("runtime_seconds"),
            "interpretation": "controlled hyperparameter screening, validation only",
        })
if summary_rows:
    controlled_summary_df = pd.DataFrame(summary_rows)
    controlled_summary_df.to_csv(TABLES_DIR / "controlled_trial_summary.csv", index=False)
    display(controlled_summary_df.head(20))
else:
    print("Controlled-trial summary waits for real run outputs.")

selected_path = CONFIGS_DIR / "06_selected_config.json"
controlled_path = CONFIGS_DIR / "controlled_selected_config.json"
ensure_selected_config_from_results(selected_path, files["batch_size"], "06_learning_rate_batch_size")

if controlled_path.exists():
    controlled_selected_config = load_json(controlled_path)
    print("Loaded controlled selected config:", controlled_path)
elif selected_path.exists():
    controlled_selected_config = load_json(selected_path)
    save_json(controlled_selected_config, controlled_path)
    print("Saved controlled selected config:", controlled_path)
else:
    controlled_selected_config = None
    print("Missing 06_selected_config.json and controlled_selected_config.json")

search_space_path = CONFIGS_DIR / "screened_hpo_search_space.json"
if controlled_selected_config and len(results) == len(files):
    hpo_space = {
        "optimizer": [controlled_selected_config["optimizer"]],
        "activation": top_values(results["activation"], "activation", 2),
        "hidden_units": top_values(results["architecture"], "hidden_units", 3),
        "dropout": [float(value) for value in top_values(results["dropout"], "dropout", 3)],
        "learning_rate": [float(value) for value in top_values(results["learning_rate"], "learning_rate", 3)],
        "batch_size": [int(value) for value in top_values(results["batch_size"], "batch_size", 3)],
        "loss": ["binary_crossentropy"],
        "threshold": [0.5],
    }
    payload = {"search_space": hpo_space, "total_configurations": domain_size(hpo_space), "optimizer_fixed_after_screening": True, "source": "controlled validation screening"}
    if payload["total_configurations"] < 40:
        print("Domain has fewer than 40 configurations; retain additional next-best screened candidates before final HPO.")
    save_json(payload, search_space_path)
    display(pd.DataFrame([(key, values) for key, values in hpo_space.items()], columns=["factor", "retained_values"]))
elif search_space_path.exists():
    print("Loaded existing HPO search space:", search_space_path)
    payload = load_json(search_space_path)
    display(pd.DataFrame([(key, values) for key, values in payload["search_space"].items()], columns=["factor", "retained_values"]))
else:
    print("HPO search space not created yet. [RESULT TO BE INSERTED AFTER FINAL RUN]")


,stage,candidate,varied_factor,validation_macro_f1,best_validation_loss,runtime_seconds,interpretation
0,baseline,fixed_reference,none,0.965466,0.081119,24.439689,"starting point, not an optimum"
1,activation,relu,activation,0.965526,0.086989,11.215436,"controlled hyperparameter screening, validatio..."
2,activation,tanh,activation,0.955892,0.103235,12.229584,"controlled hyperparameter screening, validatio..."
3,activation,elu,activation,0.966686,0.077194,16.672579,"controlled hyperparameter screening, validatio..."
4,optimizer,adam,optimizer,0.968457,0.077113,32.224538,"controlled hyperparameter screening, validatio..."
5,optimizer,rmsprop,optimizer,0.973099,0.064414,17.140474,"controlled hyperparameter screening, validatio..."
6,optimizer,sgd_momentum,optimizer,0.926774,0.154163,21.590923,"controlled hyperparameter screening, validatio..."
7,dropout,0.0,dropout,0.953164,0.117642,7.017143,"controlled hyperparameter screening, validatio..."
8,dropout,0.1,dropout,0.966449,0.070958,14.080634,"controlled hyperparameter screening, validatio..."
9,dropout,0.2,dropout,0.963307,0.080667,12.503044,"controlled hyperparameter screening, validatio..."


Saved controlled selected config: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/mlp/configs/controlled_selected_config.json


,factor,retained_values
0,optimizer,[rmsprop]
1,activation,"[elu, relu]"
2,hidden_units,"[[128, 64], [256, 128, 64], [256, 128]]"
3,dropout,"[0.1, 0.3, 0.2]"
4,learning_rate,"[0.001, 0.003, 0.0003]"
5,batch_size,"[128, 256, 64]"
6,loss,[binary_crossentropy]
7,threshold,[0.5]


## 3. Checks


In [4]:
# Purpose: Runs this notebook step and prints or saves the resulting intermediate output for review.
display(pd.DataFrame([("screening_not_global_optimisation", True), ("optimizer_fixed_for_hpo", True), ("test_metrics_computed", False)], columns=["check", "value"]))


,check,value
0,screening_not_global_optimisation,True
1,optimizer_fixed_for_hpo,True
2,test_metrics_computed,False
